# Import libraries and set configs

In [ ]:
import sys

sys.path.append("..")
sys.path.append("../..")

import ast
import json
import numpy as np
import pandas as pd


class CFG:
    select_features = True
    select_features_to_remove = True
    optimize = False
    n_repeats = 1
    n_folds = 8


# Load the train data

In [ ]:
train_df = pd.read_pickle("data/train_df.pkl")

# all data for the last 90 days are test
test_date = train_df["time"].max() - pd.to_timedelta(90, unit="D")

profitable_hours_df = pd.read_csv("data/profitable_hours.csv")
latest = profitable_hours_df.iloc[-1]
buy_hours = ast.literal_eval(latest["profitable_buy_hours"])
sell_hours = ast.literal_eval(latest["profitable_sell_hours"])

buy_mask = (train_df["ttype"] == "buy") & (train_df["time"].dt.hour.isin(buy_hours))
sell_mask = (train_df["ttype"] == "sell") & (train_df["time"].dt.hour.isin(sell_hours))
train_df = train_df[buy_mask | sell_mask].reset_index(drop=True)

# Feature selection

### Select features with BORUTA feature importance

In [ ]:
from utils.feature_selection_utils import boruta_selction

features = [
    c
    for c in train_df.columns
    if c
    not in [
        "time",
        "target",
        "ticker",
        "pattern",
        "ttype",
        "weight",
        "max_price_deviation",
        "min_price_deviation",
        "close_time",
        "first_price",
        "last_price",
    ]
]

params = {
    "boosting_type": "gbdt",
    "n_estimators": 1000,
    "learning_rate": 0.02,
    "max_depth": 6,
    "subsample": 0.7,
    "colsample_bytree": 0.7,
    "verbosity": -1,
    "importance_type": "gain",
    "objective": "binary",
    "metric": "average_precison",
    "verbose": -1,
}

if CFG.select_features:
    boruta_df_ = boruta_selction(train_df, features, params)

### Select features with permutation importance and GBM feature importance

In [ ]:
from utils.feature_selection_utils import lgbm_tuning

# load the list of Bybit tickers
with open("model/bybit_tickers.json", "r") as f:
    bybit_tickers = json.load(f)

perm_df_, feature_importances_, outer_cv_score = lgbm_tuning(
    train_df, features, params, bybit_tickers, n_folds=4, n_repeats=CFG.n_repeats, permut=True
)

### RFE feature selection

In [ ]:
from utils.feature_selection_utils import rfe_selection

rfe_df_ = rfe_selection(train_df, features)

### Combine importances and save them

In [ ]:
boruta_df_["rank"] = boruta_df_["importance"].rank()
perm_df_["rank"] = perm_df_["importance"].rank(ascending=False)
rfe_df_["rank"] = rfe_df_["importance"]
feature_importances_["rank"] = feature_importances_["Value"].rank(ascending=False)

fi = pd.concat([
    perm_df_[["Feature","rank"]], 
    feature_importances_[["Feature","rank"]], 
    rfe_df_[["Feature","rank"]],
    boruta_df_[["Feature","rank"]],
                ])
fi = fi.groupby("Feature")["rank"].sum().reset_index()
# these features are very important they must be among features anyway
for feature in ["weekday", "funding_rate", "macdhist_prev_4"]:
    fi.loc[fi["Feature"] == feature, "rank"] = fi.loc[fi["Feature"] == feature, "rank"].values[0] / 100
fi = fi.sort_values("rank").reset_index(drop=True)
fi.to_csv("model/feature_importance.csv", index=False)

# Additional feature selection

### Load best parameters from Optuna dataframe

In [ ]:
from utils.optimization_utils import load_params_from_optuna

params = load_params_from_optuna(row_num=0)
params

### Load selected features

In [ ]:
from utils.feature_selection_utils import prepare_features

if CFG.optimize:
    feature_num = 100
    corr_thresh = 0.7

if "feature_num" in params:
    if not CFG.optimize:
        feature_num = params["feature_num"]
        corr_thresh = params["corr_thresh"]

    del params["feature_num"]
    del params["corr_thresh"]

fi = pd.read_csv("model/feature_importance.csv")
features, feature_dict = prepare_features(train_df, fi, feature_num, corr_thresh)

assert len(features) == len(set(features))

display(features, len(features))


### Prepare model parameters

In [ ]:
if "max_train_size" in params:
    max_train_size = params["max_train_size"]
    del params["max_train_size"]

# set high and low bound for model predictions
# p > high_bound -> 1, p < low_bound -> 0
if "high_bound" in params:
    high_bound = params["high_bound"]
    del params["high_bound"]
    del params["low_bound"]
low_bound = 0

# add object weights
if "sample_weight" in params:
    sample_weight = params["sample_weight"]
    del params["sample_weight"]

if sample_weight == "cos":
    train_df["weight"] = train_df["time"].astype(np.int64) / int(1e6)
    train_df["weight"] = (train_df["weight"].max() - train_df["weight"]) / (train_df["weight"].max() - train_df["weight"].min()) * np.pi / 2
    train_df["weight"] = np.cos(train_df["weight"])
    sample_weight = True
elif sample_weight == "linear":
    train_df["weight"] = train_df["time"].astype(np.int64) / int(1e6)
    train_df["weight"] = (train_df["weight"].max() - train_df["weight"]) / (train_df["weight"].max() - train_df["weight"].min())
    sample_weight = True

params["objective"] = "binary"
params["verbosity"] = -1
if params["boosting_type"] != "goss":
    params["subsample_freq"] = 1
else:
    params["subsample"] = None
    params["subsample_freq"] = None
params["importance_type"] = "gain"
params["metric"] = "average_precison"

### Remove features that significantly decrease fold score

In [ ]:
from utils.feature_selection_utils import remove_feature_selection

if CFG.select_features_to_remove:
    features_to_remove = remove_feature_selection(
        train_df, test_date, features, params, high_bound, max_train_size
    )
    features_to_remove